# 08 – v2 Analysis and Figures

CPU only. This notebook reads the analysis outputs of one split, running the analysis first if it is missing, and builds every table and figure for the paper. Figures go to `results/figures_v2/<split>/` as PNG and PDF. Each figure is followed by the table it was drawn from: several palette colours are below 3:1 contrast, so the numbers must stay readable without the colours.

| § | Content | Gate |
|---|---|---|
| 1 | Verdicts | all |
| 2 | Metric audit: controls vs methods, OA vs box OA by size, headroom, conditioning | B |
| 3 | TP/FP verification: forest plot of ΔAUROC, feature-set table | G1 |
| 4 | Evidence acquisition under a budget, value per method | G2 |
| 5 | AP re-scoring | G3 |
| 6 | Abstention with conformal risk control | — |
| 7 | Subgroups | X |
| 8 | Qualitative examples: evidence rejects a confident FP / rescues a weak TP | — |
| 9 | v1 comparison rows | — |

Set `SPLIT` to `"dev"` (pilot, exploratory), `"test"` (confirmatory) or `"bdd"`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src import notebook_utils as nbu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

import matplotlib.pyplot as plt

from src import v2_figures as vf

SPLIT = "test"                     # "dev" | "test" | "bdd"
RERUN_ANALYSIS = False             # True: recompute even if the analysis exists
ENV_FILE = "cluster/cluster.env"

TASK = {"dev": "analyze-pilot", "test": "analyze-test", "bdd": "analyze-bdd"}[SPLIT]
PREFIX = {"dev": "coco_dev", "test": "coco_test", "bdd": "bdd_test"}[SPLIT]
EVIDENCE_DIR = nbu.ANALYSIS_ROOT / f"evidence_{PREFIX}"
VALIDITY_DIR = nbu.ANALYSIS_ROOT / f"metric_validity_{PREFIX}"
FIG_DIR = nbu.PROJECT_ROOT / "results" / "figures_v2" / SPLIT
env = nbu.load_cluster_env(ENV_FILE)
IMAGES_DIR = nbu.PROJECT_ROOT / (env["BDD_IMAGES"] if SPLIT == "bdd" else env["COCO_IMAGES"])
vf.apply_style()
saved = []

def show(fig, name):
    saved.extend(vf.save_figure(fig, name, FIG_DIR))
    plt.show()

In [ ]:
if RERUN_ANALYSIS or not (EVIDENCE_DIR / "summary.json").exists() or not (VALIDITY_DIR / "summary.json").exists():
    nbu.run_task(TASK, env_file=ENV_FILE)
SUMMARY = {item["detector"]: item for item in vf.load_summary(EVIDENCE_DIR)}
RUNS = sorted(SUMMARY)
print("runs:", RUNS)

## 1. Verdicts

G1 passes when ΔAUROC(S3 − S2) ≥ the gate and the paired bootstrap CI excludes 0. On `dev` this is a direction check only.

In [ ]:
display(nbu.gate_summary(EVIDENCE_DIR))
display(Markdown((EVIDENCE_DIR / "report.md").read_text()))

## 2. Metric audit (gate B)

- **Controls vs methods:** if the `bbox_prior` control scores at or above real methods on the composite, the composite is mostly measuring box geometry.
- **OA by size:** if the small-to-large drop in global OA shrinks under the box-relative protocol, that trend is a property of the step schedule, not of the explanations.

In [ ]:
for run in RUNS:
    show(vf.fig_metric_audit(VALIDITY_DIR / f"methods_{run}.csv", title=f"Composite — {run}"), f"B_audit_composite_{run}")
    table = pd.read_csv(VALIDITY_DIR / f"methods_{run}.csv")
    display(table[[c for c in ["method", "is_control", "composite", "composite_ci", "ebpg", "oa", "oa_box", "sparsity", "ebpg_gt", "explain_time", "is_fallback"] if c in table]].round(4))

In [ ]:
for run in RUNS:
    show(vf.fig_oa_by_size(VALIDITY_DIR / f"by_size_{run}.csv", title=f"OA by object size — {run}"), f"B_oa_by_size_{run}")
    display(pd.read_csv(VALIDITY_DIR / f"size_trend_{run}.csv").round(4))

In [ ]:
validity = json.loads((VALIDITY_DIR / "summary.json").read_text())
for run in RUNS:
    display(Markdown(f"**{run}** — headroom per objective (oracle − best fixed method)"))
    display(pd.read_csv(VALIDITY_DIR / f"headroom_{run}.csv").round(4))
    conditioning = VALIDITY_DIR / f"conditioning_{run}.csv"
    if conditioning.exists():
        display(Markdown("conditioning deltas (with − without box conditioning)"))
        display(pd.read_csv(conditioning).round(4))
    agreement = validity.get(run, {}).get("pseudo_gt_agreement", {})
    if agreement:
        print("method ranking by detector-box vs GT-box EBPG (Spearman):", round(agreement["ranking_spearman_det_vs_gt_box"], 3))

## 3. TP/FP verification (G1)

- **Gate row:** explanation evidence vs output + gradient.
- **Placebo rows:** maps without model evidence (controls, shuffled maps) scored by the same evaluator. If a placebo reaches the same gain, the signal comes from perturbation probing of the detector, not from the explanation's content. Report that honestly.

In [ ]:
for run in RUNS:
    show(vf.fig_g1_forest(SUMMARY[run]), f"G1_forest_{run}")
    metrics = pd.read_csv(EVIDENCE_DIR / f"g1_metrics_{run}.csv")
    display(metrics[["set", "num_features", "auroc", "auprc", "fpr_at_95tpr", "ece"]].round(4))

## 4. Evidence acquisition under a budget (G2)

`learned_voi` ranks detections by the predicted value of computing their explanation evidence. Gate: at a 50% budget it keeps ≥ 70% of the full-evidence gain and beats `random`. The value table shows which explainer is worth its cost; that per-cost choice replaces the v1 idea of choosing the explainer per detection.

In [ ]:
for run in RUNS:
    show(vf.fig_g2_acquisition(EVIDENCE_DIR / f"g2_acquisition_{run}.csv", title=f"G2 — {run}"), f"G2_acquisition_{run}")
    display(pd.read_csv(EVIDENCE_DIR / f"g2_acquisition_{run}.csv").pivot_table(index="budget", columns="policy", values="auroc").round(4))
    display(nbu.suggest_lite_methods(EVIDENCE_DIR, run).round(4))

## 5. AP re-scoring (G3)

Detections are re-scored with the out-of-fold P(TP) and re-matched to GT. Unexplained detections keep the output-feature score.

In [ ]:
rows = []
for run in RUNS:
    g3 = SUMMARY[run]["g3"]
    for name, value in g3["ap50_delta_vs_confidence"].items():
        rows.append({"run": run, "score": name, "AP50": g3["ap50"][name], "AP50 (confidence)": g3["ap50"]["confidence"],
                     "delta": value["delta"], "ci_low": value["ci_low"], "ci_high": value["ci_high"],
                     "AP75": g3.get("ap75", {}).get(name)})
display(pd.DataFrame(rows).round(4))

## 6. Abstention with conformal risk control

The risk is the fraction of accepted false positives per image, with a finite-sample guarantee on the calibration half, averaged over 20 image splits. Higher TP coverage at the same risk means fewer correct detections are needlessly rejected. The per-subgroup coverage columns show who pays for abstention.

In [ ]:
for run in RUNS:
    table = pd.read_csv(EVIDENCE_DIR / f"abstention_crc_{run}.csv")
    for alpha in sorted(table["alpha"].unique()):
        show(vf.fig_abstention(EVIDENCE_DIR / f"abstention_crc_{run}.csv", alpha, title=f"Abstention at risk ≤ {alpha} — {run}"), f"abstention_{run}_{alpha}")
    display(table.round(4))

## 7. Subgroups

AUROC per COCO size bin, class-frequency tercile and, for BDD100K, time of day and weather. Subgroups with fewer than 50 detections are omitted.

In [ ]:
for run in RUNS:
    path = EVIDENCE_DIR / f"subgroup_auroc_{run}.csv"
    table = pd.read_csv(path)
    for column in table["column"].unique():
        show(vf.fig_subgroup_auroc(path, column, title=f"AUROC by {column.replace('attr__', '')} — {run}"), f"subgroup_{run}_{column}")
    display(table.round(4))

## 8. Qualitative examples

These come from the out-of-fold predictions:
- **(a)** confident detections (≥ 0.7) that are FPs and that the evidence model rejects (P(TP) ≤ 0.3);
- **(b)** weak detections (≤ 0.4) that are TPs and that it rescues (P(TP) ≥ 0.7).

This needs saved maps (`save_maps_for`) and the images.

In [ ]:
from src.utils import load_image

for run in RUNS:
    predictions = pd.read_csv(EVIDENCE_DIR / f"g1_predictions_{run}.csv")
    run_dir = nbu.EVIDENCE_ROOT / run
    cases = {
        "confident FP rejected": predictions[(predictions.confidence >= 0.7) & (~predictions.is_tp) & (predictions.p_S3_explanation <= 0.3)],
        "weak TP rescued": predictions[(predictions.confidence <= 0.4) & (predictions.is_tp) & (predictions.p_S3_explanation >= 0.7)],
    }
    for label, rows in cases.items():
        display(Markdown(f"**{run} — {label}: {len(rows)} detections**"))
        for index, row in enumerate(rows.head(3).itertuples()):
            npz = run_dir / "maps" / f"{Path(row.image).stem}.npz"
            if not npz.exists() or not (IMAGES_DIR / row.image).exists():
                continue
            image = load_image(str(IMAGES_DIR / row.image))
            maps = vf.load_saved_maps(npz, int(row.det_id), image.shape)
            title = f"{row.image} · {row.class_name} · conf {row.confidence:.2f} → P(TP) {row.p_S3_explanation:.2f} · {'TP' if row.is_tp else 'FP'}"
            show(vf.show_detection_maps(image, maps, json.loads(row.bbox), title=title), f"qualitative_{run}_{label.replace(' ', '_')}_{index}")

## 9. v1 comparison rows

Per-image composite means from `results/v2_legacy/`. There are two versions: images without detections scored as 0 (v1 convention), and valid images only.

In [ ]:
rows = []
legacy_root = nbu.PROJECT_ROOT / "results" / "v2_legacy"
for path in sorted(legacy_root.glob("**/*.csv")):
    if path.name == "detections.csv":
        continue
    frame = pd.read_csv(path)
    if "composite_score" not in frame:
        continue
    valid = frame[frame["num_detections"] > 0]
    rows.append({"source": str(path.relative_to(legacy_root)), "detector": frame["detector"].iloc[0] if "detector" in frame else None,
                 "images": len(frame), "composite (empty=0)": frame["composite_score"].mean(),
                 "composite (valid only)": valid["composite_score"].mean(), "mean OA": frame["mean_oa"].mean()})
display(pd.DataFrame(rows).round(4) if rows else "No legacy rows yet (stage 5).")

## Saved figures

In [ ]:
print(f"{len(saved)} files in {FIG_DIR}")
for path in saved:
    print(" ", path.name)